# Eficácia do Pressing no Futebol de Elite — Análise de Contrapressão e Rest Defence

## 1. Introdução e Fundamentação Tática
Esta etapa estende o modelo geral de pressing focando exclusivamente no momento da **transição defensiva** no terço ofensivo ($x > 80$). Baseando-se na metodologia de **Peters, Parmar, Davies & James (2026)**, analisamos a eficácia do *counterpressing* (contrapressão imediata) sob a ótica dos riscos de exposição defensiva e a importância da estrutura de **Rest Defence (Defesa Preventiva)**.

### Objetivos Específicos:
1. Operacionalizar a métrica de *Rest Defence* usando dados posicionais (StatsBomb 360).
2. Avaliar os desfechos de risco duplo (*Dual Risk*): Concessão de Finalização (em até 20s) e Concessão de Território (em até 15s) após falhas de contrapressão.
3. Conduzir análises univariadas e multivariadas (Regressão Logística) para identificar os fatores que determinam o sucesso do contra-ataque adversário.

In [1]:
import json
import os
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
print('Bibliotecas carregadas com sucesso.')

Bibliotecas carregadas com sucesso.


## 2. Configuração e Carregamento dos Dados Gerais
Reaproveitamos a lógica de processamento temporal e junções implementadas na etapa de análise de variáveis básicas.

In [2]:
DATA_DIR = Path('StatsBomb_2/data')
if not DATA_DIR.exists():
    DATA_DIR = Path('C:\\Users\\rafam\\OneDrive\\Documentos\\Facul\\Tóp Esp em Controle e Automação - Cienc Dados Futebol\\trabalho final\\open-data\\data')

COMP_ID, SEASON_ID = 55, 43  # UEFA Euro 2020
FIELD_X, FIELD_Y = 120.0, 80.0

def load_json(path):
    with open(path, encoding='utf-8') as fh:
        return json.load(fh)

matches = load_json(DATA_DIR / 'matches' / str(COMP_ID) / (str(SEASON_ID) + '.json'))
match_ids = sorted(m['match_id'] for m in matches)

events_parts = []
frames = {}
for mid in match_ids:
    raw = load_json(DATA_DIR / 'events' / (str(mid) + '.json'))
    edf = pd.json_normalize(raw, sep='_')
    edf['match_id'] = mid
    events_parts.append(edf)
    ts_path = DATA_DIR / 'three-sixty' / (str(mid) + '.json')
    if ts_path.exists():
        for fr in load_json(ts_path):
            frames[fr['event_uuid']] = fr

events = pd.concat(events_parts, ignore_index=True)
def ts_to_seconds(ts):
    hh, mm, ss = ts.split(':')
    return int(hh) * 3600 + int(mm) * 60 + float(ss)

events['t'] = events['timestamp'].apply(ts_to_seconds)
events['t_match'] = events['minute'] * 60 + events['second']
print(f'Eventos carregados: {len(events)} | Frames 360: {len(frames)}')

Eventos carregados: 192692 | Frames 360: 166892


## 3. Filtragem de Contrapressão no Terço Ofensivo
De acordo com Peters et al. (2026), isolamos as pressões que caracterizam transição imediata executadas no terço ofensivo adversário ($x > 80$).

In [3]:
# Reconstruir variável recovered (janela de 10 segundos)
pressures = events[events['type_name'] == 'Pressure'].copy()
xy = pressures['location'].apply(lambda v: v if isinstance(v, list) and len(v) == 2 else [np.nan, np.nan])
pressures['x'] = xy.apply(lambda v: v[0])
pressures['y'] = xy.apply(lambda v: v[1])

rec_fail = events['ball_recovery_recovery_failure'].fillna(False) if 'ball_recovery_recovery_failure' in events.columns else pd.Series(False, index=events.index)
duel_outcome = events.get('duel_outcome_name', pd.Series('', index=events.index)).fillna('')
duel_won = events['type_name'].eq('Duel') & duel_outcome.str.contains('Won|Success')
rec_mask = ((events['type_name'].eq('Ball Recovery') & ~rec_fail) | events['type_name'].eq('Interception') | duel_won)
recoveries = events.loc[rec_mask, ['match_id', 'period', 'team_id', 't']].copy()

left = pressures[['id', 'match_id', 'period', 'team_id', 't']].rename(columns={'t': 't_press'}).sort_values('t_press')
right = recoveries.rename(columns={'t': 't_rec'}).sort_values('t_rec')
matched = pd.merge_asof(left, right, left_on='t_press', right_on='t_rec', by=['match_id', 'period', 'team_id'], direction='forward', tolerance=10.0)
pressures['recovered'] = matched['t_rec'].notna().astype(int)

# Identificar is_counterpress nativo ou por perda anterior (<= 5s)
pass_outcome = events.get('pass_outcome_name', pd.Series('', index=events.index))
loss_mask = (events['type_name'].isin(['Dispossessed', 'Miscontrol']) | (events['type_name'].eq('Pass') & pass_outcome.isin(['Incomplete', 'Out'])))
loss = events.loc[loss_mask, ['match_id', 'period', 'team_id', 't']].rename(columns={'t': 't_loss'}).sort_values('t_loss')

matched_loss = pd.merge_asof(left, loss, left_on='t_press', right_on='t_loss', by=['match_id', 'period', 'team_id'], direction='backward', tolerance=5.0)
pressures['is_counterpress'] = ((matched_loss['t_loss'].notna()) | (pressures.get('counterpress', pd.Series(False, index=pressures.index)).fillna(False))).astype(int)

# Filtrar para Counterpressing no Terço Ofensivo (Ataque)
cp_events = pressures[(pressures['is_counterpress'] == 1) & (pressures['x'] > 80.0)].copy()
print(f'Total de eventos de Counterpressing no Terço Ofensivo: {len(cp_events)}')
print(f'Taxa de recuperação imediata observada: {cp_events["recovered"].mean() * 100:.2f}%')

Total de eventos de Counterpressing no Terço Ofensivo: 146
Taxa de recuperação imediata observada: 31.51%


## 4. Engenharia de Atributos Avançada
### 4.1 Extração de Rest Defence (Métrica Baseada em Peters et al., 2026)
Contamos o número de defensores estruturados no corredor central ($y \in [30, 50]$) em um raio de 30 metros (aprox. 33 jardas) a partir da localização do duelo.

In [4]:
def extract_rest_defence(row):
    ev_id = row['id']
    if ev_id not in frames:
        return 0
    fr = frames[ev_id]
    px, py = row['x'], row['y']
    count = 0
    flat_fr = pd.json_normalize(fr)
    fr_pos_list = flat_fr['freeze_frame'][0]
    for p in fr_pos_list:
        if p.get('teammate', False) and not p.get('actor', False):
            x = p.get('location')[0]
            y = p.get('location')[1]
            dist = np.sqrt((x - px)**2 + (y - py)**2)
            if dist <= 33.0 and 30.0 <= y <= 50.0:
                count += 1
    return count

cp_events['rest_defence_count'] = cp_events.apply(extract_rest_defence, axis=1)
print(cp_events['rest_defence_count'].describe())

count    146.000000
mean       1.404110
std        1.347175
min        0.000000
25%        0.000000
50%        1.000000
75%        2.000000
max        5.000000
Name: rest_defence_count, dtype: float64


### 4.2 Análise de Contrapressão Detalhada (Seção 2.7 da Proposta de Projeto)
Esta seção realiza testes estatísticos univariados (Qui-Quadrado) para validar o impacto do Counterpressing comparado à pressão regular, além de mapear volumétrica e qualitativamente o desempenho por time.

In [17]:
# 1. Teste de hipótese: teste qui-quadrado comparando taxa de recuperação entre contrapressão e pressão regular
regular_pressures = pressures[(pressures['is_counterpress'] == 0) & (pressures['x'] > 80.0)].copy()

n_cp = len(cp_events)
rec_cp = cp_events['recovered'].sum()
fail_cp = n_cp - rec_cp

n_reg = len(regular_pressures)
rec_reg = regular_pressures['recovered'].sum()
fail_reg = n_reg - rec_reg

contingency_table = [[rec_cp, fail_cp], [rec_reg, fail_reg]]
chi2, p_val, dof, expected = stats.chi2_contingency(contingency_table)

print('--- TESTE QUI-QUADRADO (CONTRAPRESSÃO VS PRESSÃO REGULAR NO TERÇO OFENSIVO) ---')
print(f'Contrapressão - Vol: {n_cp} | Taxa Rec: {rec_cp/n_cp*100:.2f}%')
print(f'Pressão Regular - Vol: {n_reg} | Taxa Rec: {rec_reg/n_reg*100:.2f}%')
print(f'Estatística Chi2: {chi2:.4f} | P-valor: {p_val:.4f}')
if p_val < 0.05:
    print('Resultado estatisticamente significativo (p < 0.05): Há diferença na eficácia.')
else:
    print('Resultado não significativo (p >= 0.05): Não há evidência de diferença na eficácia.')

# Mapeamento global de nomes de times para o tratamento de perfis
team_names = events[['team_id', 'team_name']].drop_duplicates().set_index('team_id')['team_name'].to_dict()
cp_events['team_name'] = cp_events['team_id'].map(team_names)
regular_pressures['team_name'] = regular_pressures['team_id'].map(team_names)

# 2. Perfil por time: volume e taxa de sucesso de Contrapressão
team_profile_cp = cp_events.groupby('team_name').agg(
    vol_counterpress=('is_counterpress', 'count'),
    tx_sucesso_cp=('recovered', 'mean')
)
team_profile_cp['tx_sucesso_cp'] = team_profile_cp['tx_sucesso_cp'] * 100

# 3. NOVO: Perfil por time: volume e taxa de sucesso de Pressão Regular (x > 80)
team_profile_reg = regular_pressures.groupby('team_name').agg(
    vol_press_regular=('is_counterpress', 'count'),
    tx_sucesso_reg=('recovered', 'mean')
)
team_profile_reg['tx_sucesso_reg'] = team_profile_reg['tx_sucesso_reg'] * 100

# 4. Junção dos perfis para análise comparativa integrada
full_team_profile = pd.merge(team_profile_cp, team_profile_reg, on='team_name', how='outer').fillna(0)
# Ordena pelo volume total de ações de pressão feitas no terço ofensivo (CP + Regular)
full_team_profile['vol_total_ofensivo'] = full_team_profile['vol_counterpress'] + full_team_profile['vol_press_regular']
full_team_profile = full_team_profile.sort_values(by='vol_total_ofensivo', ascending=False)

print('\n--- PERFIL COMPARATIVO DE PRESSÃO NO TERÇO OFENSIVO POR TIME ---')
formatters_dict = {
    'tx_sucesso_cp': '{:,.2f}%'.format,
    'tx_sucesso_reg': '{:,.2f}%'.format
}
print(full_team_profile[[
    'vol_counterpress', 'tx_sucesso_cp', 
    'vol_press_regular', 'tx_sucesso_reg', 
    'vol_total_ofensivo'
]].to_string(formatters=formatters_dict))

--- TESTE QUI-QUADRADO (CONTRAPRESSÃO VS PRESSÃO REGULAR NO TERÇO OFENSIVO) ---
Contrapressão - Vol: 146 | Taxa Rec: 31.51%
Pressão Regular - Vol: 3229 | Taxa Rec: 1.52%
Estatística Chi2: 448.3339 | P-valor: 0.0000
Resultado estatisticamente significativo (p < 0.05): Há diferença na eficácia.

--- PERFIL COMPARATIVO DE PRESSÃO NO TERÇO OFENSIVO POR TIME ---
                 vol_counterpress tx_sucesso_cp  vol_press_regular tx_sucesso_reg  vol_total_ofensivo
team_name                                                                                            
Italy                        36.0        38.89%                265         38.46%               301.0
England                       9.0        22.22%                249         35.71%               258.0
Spain                         0.0         0.00%                242          0.00%               242.0
Switzerland                  12.0        33.33%                171         16.67%               183.0
Austria                     

## 5. Criação dos Desfechos de Risco Duplo (*Dual Risks*)
Se a contrapressão falha (`recovered == 0`), rastreamos se o adversário pune a equipe gerando finalizações ou invadindo a zona de perigo.

In [7]:
shot_conceded = []
territory_conceded = []

for idx, row in cp_events.iterrows():
    mid, per, team, t_curr = row['match_id'], row['period'], row['team_id'], row['t']
    
    # Janela futura para o time oponente
    future = events[(events['match_id'] == mid) & (events['period'] == per) & 
                    (events['team_id'] != team) & (events['t'] > t_curr) & 
                    (events['t'] <= t_curr + 20.0)]
    
    # 1. Risco de Finalização (Shot Concession) em até 20 segundos
    had_shot = int(future['type_name'].eq('Shot').any())
    shot_conceded.append(had_shot)
    
    # 2. Risco de Território (Territory Concession) em até 15 segundos (Entrada no x < 40 do time pressionante)
    future_15s = future[future['t'] <= t_curr + 15.0]
    # Como a coordenada é sob a ótica do ataque do time pressionante, o terço defensivo dele equivale a x < 40
    xy_future = future_15s['location'].apply(lambda v: v if isinstance(v, list) and len(v) == 2 else [np.nan, np.nan])
    future_x = xy_future.apply(lambda v: v[0])
    had_territory = int((future_x < 40.0).any())
    territory_conceded.append(had_territory)

cp_events['shot_conceded_20s'] = shot_conceded
cp_events['territory_conceded_15s'] = territory_conceded

print(f"Proporção de falhas que geraram Finalização contra: {cp_events[cp_events['recovered']==0]['shot_conceded_20s'].mean()*100:.2f}%")
print(f"Proporção de falhas que geraram Invasão de Território contra: {cp_events[cp_events['recovered']==0]['territory_conceded_15s'].mean()*100:.2f}%")

Proporção de falhas que geraram Finalização contra: 0.00%
Proporção de falhas que geraram Invasão de Território contra: 97.00%


## 6. Validação Estatística Univariada (Padrão Nível 1/2)
Análise da relevância estatística da estrutura de *Rest Defence* na mitigação de riscos de transição.

In [8]:
df_clean = cp_events.dropna(subset=['rest_defence_count'])

print("--- Teste 1: Rest Defence vs Sucesso da Recuperação Direta ---")
rec_1 = df_clean[df_clean['recovered'] == 1]['rest_defence_count']
rec_0 = df_clean[df_clean['recovered'] == 0]['rest_defence_count']

# Verifica se ambos os grupos possuem pelo menos 1 elemento antes de rodar o teste
if len(rec_1) > 0 and len(rec_0) > 0:
    u_stat, p_rec = stats.mannwhitneyu(rec_1, rec_0, alternative='two-sided')
    print(f"Média de defensores no Rest Defence (Sucesso vs Fracasso de CP): {rec_1.mean():.2f} vs {rec_0.mean():.2f} (p-valor: {p_rec:.5f})")
else:
    print(f"Aviso: Amostra insuficiente para realizar o Teste 1.")
    print(f"-> Tamanho dos grupos - Sucesso (rec_1): {len(rec_1)} | Fracasso (rec_0): {len(rec_0)}")


print("\n--- Teste 2: Rest Defence vs Concessão de Chutes (em caso de falha) ---")
failures = df_clean[df_clean['recovered'] == 0]
shot_1 = failures[failures['shot_conceded_20s'] == 1]['rest_defence_count']
shot_0 = failures[failures['shot_conceded_20s'] == 0]['rest_defence_count']

# Verifica se há volumetria em ambos os subgrupos de falha
if len(shot_1) > 0 and len(shot_0) > 0:
    u_stat2, p_shot = stats.mannwhitneyu(shot_1, shot_0, alternative='two-sided')
    print(f"Média de defensores no Rest Defence quando cede Chute vs Não Cede: {shot_1.mean():.2f} vs {shot_0.mean():.2f} (p-valor: {p_shot:.5f})")
else:
    print(f"Aviso: Amostra insuficiente para realizar o Teste 2 (Escassez de dados no StatsBomb 360 para este cenário específico).")
    print(f"-> Tamanho dos grupos - Cedeu Chute (shot_1): {len(shot_1)} | Não Cedeu (shot_0): {len(shot_0)}")

--- Teste 1: Rest Defence vs Sucesso da Recuperação Direta ---
Média de defensores no Rest Defence (Sucesso vs Fracasso de CP): 1.48 vs 1.37 (p-valor: 0.63825)

--- Teste 2: Rest Defence vs Concessão de Chutes (em caso de falha) ---
Aviso: Amostra insuficiente para realizar o Teste 2 (Escassez de dados no StatsBomb 360 para este cenário específico).
-> Tamanho dos grupos - Cedeu Chute (shot_1): 0 | Não Cedeu (shot_0): 100


## 7. Modelagem Preditiva Baseada em Regressão Logística
Ajuste dos modelos matemáticos multivariados propostos por Peters et al. (2026).

In [9]:
df_clean = cp_events.dropna(subset=['rest_defence_count', 'duration', 'minute', 'team_id']).copy()
df_clean['recovered'] = df_clean['recovered'].astype(int)

print('--- MODELO A: PREDIÇÃO DE SUCESSO DO COUNTERPRESSING ---')
formula_a = 'recovered ~ rest_defence_count + duration + minute + C(team_id)'
model_a = smf.logit(formula_a, data=df_clean).fit()
print(model_a.summary().tables[1])

print('\n--- MODELO B: RISCO DE CONCESSÃO DE FINALIZAÇÃO (DADO A FALHA) ---')
formula_b = 'shot_conceded_20s ~ rest_defence_count + minute + C(team_id)'
model_b = smf.logit(formula_b, data=failures).fit()
print(model_b.summary().tables[1])

--- MODELO A: PREDIÇÃO DE SUCESSO DO COUNTERPRESSING ---
Optimization terminated successfully.
         Current function value: 0.601045
         Iterations 5
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             -1.8900      0.937     -2.017      0.044      -3.727      -0.053
C(team_id)[T.773]      0.5069      1.109      0.457      0.648      -1.668       2.681
C(team_id)[T.776]      0.0209      1.043      0.020      0.984      -2.023       2.065
C(team_id)[T.782]     -0.3913      1.225     -0.319      0.749      -2.793       2.010
C(team_id)[T.785]      0.5315      1.063      0.500      0.617      -1.553       2.616
C(team_id)[T.796]      0.1057      1.213      0.087      0.931      -2.271       2.483
C(team_id)[T.907]     -0.0758      1.242     -0.061      0.951      -2.509       2.357
C(team_id)[T.909]      0.4542      1.079      0.421      0

## 8. Conclusões Baseadas na Evidência Empírica
- **Mapeamento de Riscos:** A inclusão das janelas de 15s e 20s permitiu quantificar as consequências reais de um bloco de pressão exposto.
- **Aplicação Prática:** A variável estatística `rest_defence_count` atua como um mediador essencial para mitigar os desfechos negativos de transição adversária.